# AF2 spectral — staged global decision\nAttach the seven individual arm output ZIPs plus the private core dataset. No training and no test access.

In [ ]:
import json, os, shutil, subprocess, sys\nfrom pathlib import Path\nWORK=Path('/kaggle/working'); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input'); OUT=WORK/'af2-spectral-factorization-v1'; REPORTS=OUT/'val_reports'\nif REPO.exists(): shutil.rmtree(REPO)\nfor _ in range(3):\n if subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]).returncode==0: break\n if REPO.exists(): shutil.rmtree(REPO)\nelse: raise RuntimeError('git clone gagal tiga kali')\nsubprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True); os.chdir(REPO); REPORTS.mkdir(parents=True,exist_ok=True)\nfrom coffee_detector.experiments.run_faruq_v3_af2_spectral_decision import run_spectral_decision\narms=('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM','PCG1','WAV1')\nfor arm in arms:\n matches=sorted(INPUT.rglob(f'{arm}_seed42_result.json'))\n if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu hasil {arm}; ditemukan {matches}')\n payload=json.loads(matches[0].read_text()); assert payload['arm']==arm and payload['test_images_accessed'] is False\n shutil.copy2(matches[0],REPORTS/matches[0].name)\nbaseline=sorted(INPUT.rglob('lfdet_afab_seed42_screening.json'))\nif len(baseline)!=1: raise FileNotFoundError(f'Harus ada tepat satu evidence AF2: {baseline}')\nstage1=run_spectral_decision(OUT,baseline[0],stage='stage1'); stage2=run_spectral_decision(OUT,baseline[0],stage='stage2'); global_result=run_spectral_decision(OUT,baseline[0],stage='global')\nassert global_result['test_opened'] is False; print(json.dumps(global_result,indent=2))\nprint('DOWNLOAD SEBELUM STOP SESSION:',shutil.make_archive('/kaggle/working/af2-spectral-global-decision','zip',OUT))